# CMS Claims Analytics Platform — End-to-End Walkthrough

**Portfolio demo for a Principal Software Engineer interview.**  
Built on the [CMS DE-SynPUF](https://www.cms.gov/data-research/statistics-trends-and-reports/medicare-claims-synthetic-public-use-files/cms-2008-2010-data-entrepreneurs-synthetic-public-use-file-de-synpuf) synthetic Medicare claims dataset (2008–2010).

This notebook walks the full data pipeline from raw CSV ingestion through star schema construction, analytical SQL, ML risk scoring, and AI-driven care-gap explanation — ending with a concrete V0 → V2 scaling discussion.

---

## Architecture Tiers

| Tier | What it is | Key technology |
|------|------------|----------------|
| **V0** | Batch analytics core: CSV → DuckDB star schema + SQL query library | DuckDB, Polars, scikit-learn |
| **V1** | Served platform: FastAPI over analytics + AI-driven risk scoring + Ollama care-gap explainer | FastAPI, Pydantic v2, Ollama |
| **V2** | Scale & resiliency: Postgres migration path, Kafka streaming ingestion, HA, multi-tenancy | Postgres, Kafka, Docker Swarm/K8s |

> The V2 design drove every V0 abstraction decision. DuckDB was chosen knowing the swap point to Postgres. Chunked ingestion was built with the Kafka seam annotated in the code. The subsample-list config exists because V2 sharding will be by beneficiary hash. **V0 is not a prototype — it is a deliberately simple foundation with a clear migration path.**

In [ ]:
# Setup: add src/ to path, import core libraries
import sys
import os

# Add src layout to path for local development
repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
src_path = os.path.join(repo_root, "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import duckdb
import polars as pl

print(f"DuckDB: {duckdb.__version__}")
print(f"Polars: {pl.__version__}")
print(f"Python: {sys.version.split()[0]}")

---

## Section 1: Data Acquisition

The CMS DE-SynPUF dataset is organized into **20 subsamples**, each containing **8 zip files**:

| File | Contents | Years |
|------|----------|-------|
| Beneficiary Summary (×3) | Demographics, chronic conditions, reimbursements | 2008, 2009, 2010 |
| Inpatient Claims (×1) | Hospital admissions, DRG codes, LOS | 2008–2010 |
| Outpatient Claims (×1) | Facility outpatient visits | 2008–2010 |
| Carrier Claims (×2) | Physician/supplier claims (A and B files) | 2008–2010 |
| Prescription Drug Events (×1) | Part D pharmacy fills | 2008–2010 |

**Key design decisions:**
- **Explicit schemas only** — `all_varchar=true` on ingest, type casts happen in the transform layer. No silent type inference that breaks on new data.
- **Idempotent downloads** — files are skipped if already present on disk; the manifest JSON records provenance.
- **Subsample-list config** — `settings.subsamples = [1]` for dev, `[1..20]` for production. Zero code changes to scale up.
- The `data/` directory is gitignored; the dataset is synthetic but handled as PHI throughout.

In [ ]:
# Show the 8 canonical file templates for subsample 1
from cms_platform.ingest.download import file_names_for_sample

print("Files for subsample 1:")
for f in file_names_for_sample(1):
    print(f"  {f}")

---

## Section 2: Star Schema

Raw CSVs land in five `raw_*` tables (all VARCHAR). The transform layer promotes them into a **star schema**:

### Dimension tables
| Table | Grain | Key design note |
|-------|-------|-----------------|
| `dim_date` | Calendar day 2007–2011 | Pre-populated; eliminates date parsing in every query |
| `dim_beneficiary` | (beneficiary_id, claim_year) | SCD-lite — one row per year to capture chronic condition changes |
| `dim_provider` | Provider NPI/number | Deduplicated across inpatient + outpatient claims |
| `dim_diagnosis` | ICD-9 code | Union across all claim types |

### Fact tables
| Table | Grain | Source |
|-------|-------|--------|
| `fact_inpatient` | One row per inpatient claim | `raw_inpatient` |
| `fact_outpatient` | One row per outpatient claim | `raw_outpatient` |
| `fact_carrier` | One row per carrier claim (LINE amounts summed) | `raw_carrier` |
| `fact_pde` | One row per prescription drug event | `raw_pde` |

### Unified view
`fact_claim_line` — UNION ALL across all four fact tables with a `claim_type` discriminator. Enables cross-type analytics in a single query.

All populate functions use `NOT EXISTS` guards — the schema build is fully idempotent.

**V2 seam:** At V2, each fact table will be hash-partitioned by `(claim_year, beneficiary_id_hash % N)` so intra-shard joins never cross node boundaries. The seam annotation is in `schema/transforms.py`.

In [ ]:
import duckdb
from cms_platform.ingest.load import _ensure_raw_tables
from cms_platform.schema.transforms import build_star_schema
from cms_platform.common.config import Settings

# In-memory DuckDB — no file I/O required for this walkthrough
conn = duckdb.connect()
_ensure_raw_tables(conn)

# --- Insert fixture: 1 beneficiary (2008) ---
conn.execute("""
    INSERT INTO raw_beneficiary (
        DESYNPUF_ID, BENE_BIRTH_DT, BENE_DEATH_DT, BENE_SEX_IDENT_CD, BENE_RACE_CD,
        BENE_ESRD_IND, SP_STATE_CODE, BENE_COUNTY_CD,
        BENE_HI_CVRAGE_TOT_MONS, BENE_SMI_CVRAGE_TOT_MONS, BENE_HMO_CVRAGE_TOT_MONS, PLAN_CVRG_MOS_NUM,
        SP_ALZHDMTA, SP_CHF, SP_CHRNKIDN, SP_CNCR, SP_COPD, SP_DEPRESSN,
        SP_DIABETES, SP_ISCHMCHT, SP_OSTEOPRS, SP_RA_OA, SP_STRKETIA,
        MEDREIMB_IP, BENRES_IP, PPPYMT_IP, MEDREIMB_OP, BENRES_OP, PPPYMT_OP,
        MEDREIMB_CAR, BENRES_CAR, PPPYMT_CAR,
        _claim_year, _source_file
    ) VALUES (
        '00013D2EFD8E45D1', '19320101', '', '1', '1',
        '0', '01', '001',
        '12', '12', '0', '12',
        '0', '1', '0', '0', '0', '0',
        '1', '0', '0', '0', '0',
        '1500.00', '0', '0', '500.00', '0', '0',
        '300.00', '0', '0',
        '2008', 'notebook'
    )
""")

# --- Insert fixture: 1 inpatient claim ---
conn.execute("""
    INSERT INTO raw_inpatient (
        DESYNPUF_ID, CLM_ID, SEGMENT, CLM_FROM_DT, CLM_THRU_DT,
        PRVDR_NUM, AT_PHYSN_NPI, OP_PHYSN_NPI, OT_PHYSN_NPI,
        CLM_PMT_AMT, NCH_PRMRY_PYR_CLM_PD_AMT, NCH_BENE_IP_DDCTBL_AMT,
        NCH_BENE_PTA_COINSRNC_LBLTY_AM, NCH_BENE_BLOOD_DDCTBL_LBLTY_AM,
        CLM_UTLZTN_DAY_CNT, NCH_BENE_DSCHRG_DT, CLM_DRG_CD,
        ICD9_DGNS_CD_1, _source_file
    ) VALUES (
        '00013D2EFD8E45D1', '2ADE5009B6E79B14', '1', '20080301', '20080305',
        'P00123456', '1234567890', '1234567891', '1234567892',
        '5000.00', '0', '1068.00',
        '0', '0',
        '4', '20080305', '291',
        '25000', 'notebook'
    )
""")

# --- Insert fixture: 1 outpatient claim ---
conn.execute("""
    INSERT INTO raw_outpatient (
        DESYNPUF_ID, CLM_ID, SEGMENT, CLM_FROM_DT, CLM_THRU_DT,
        PRVDR_NUM, AT_PHYSN_NPI, OP_PHYSN_NPI, OT_PHYSN_NPI,
        NCH_BENE_BLOOD_DDCTBL_LBLTY_AM, CLM_PMT_AMT, NCH_PRMRY_PYR_CLM_PD_AMT,
        NCH_BENE_PTB_DDCTBL_AMT, NCH_BENE_PTB_COINSRNC_AMT,
        ADMTNG_ICD9_DGNS_CD, ICD9_DGNS_CD_1, _source_file
    ) VALUES (
        '00013D2EFD8E45D1', '3BF7201C9A4E8C25', '1', '20080601', '20080601',
        'P00123456', '1234567890', '1234567891', '1234567892',
        '0', '200.00', '0',
        '0', '0',
        '25000', '25000', 'notebook'
    )
""")

# --- Insert fixture: 1 carrier claim ---
conn.execute("""
    INSERT INTO raw_carrier (
        DESYNPUF_ID, CLM_ID, CLM_FROM_DT, CLM_THRU_DT,
        ICD9_DGNS_CD_1, HCPCS_CD_1, LINE_NCH_PMT_AMT_1, _source_file
    ) VALUES (
        '00013D2EFD8E45D1', '4CF8312DAB5F9D36', '20080901', '20080901',
        '25000', 'G0010', '150.00', 'notebook'
    )
""")

# --- Insert fixture: 1 prescription drug event ---
conn.execute("""
    INSERT INTO raw_pde (
        DESYNPUF_ID, PDE_ID, SRVC_DT, PROD_SRVC_ID,
        QTY_DSPNSD_NUM, DAYS_SUPLY_NUM, PTNT_PAY_AMT, TOT_RX_CST_AMT, _source_file
    ) VALUES (
        '00013D2EFD8E45D1', '5DA9423EBC607E47', '20081001', 'METFORMIN',
        '90', '90', '5.00', '25.00', 'notebook'
    )
""")

# Build the full star schema from the raw tables
build_star_schema(conn, Settings())

# Row counts for all 8 tables + unified view
print("Star schema row counts:")
star_tables = [
    "dim_date", "dim_beneficiary", "dim_provider", "dim_diagnosis",
    "fact_inpatient", "fact_outpatient", "fact_carrier", "fact_pde",
]
for t in star_tables:
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t:<22} {n:>6} rows")

view_count = conn.execute("SELECT COUNT(*) FROM fact_claim_line").fetchone()[0]
print(f"  {'fact_claim_line (view)':<22} {view_count:>6} rows")

---

## Section 3: Analytical Queries

Five purpose-built SQL queries cover the core analytics use cases. Each query lives in `sql/analytics/` with a standard header block documenting the business question, SQL technique, and V2 scaling note.

| Query | Business question | SQL technique |
|-------|-------------------|---------------|
| `readmission_30day` | What fraction of inpatient discharges result in readmission within 30 days? | LAG window function + date range self-join |
| `cohort_segmentation` | How large are chronic-condition cohorts and what is their comorbidity burden? | CASE/FILTER aggregation, comorbidity count |
| `cost_benchmarking` | Where does each beneficiary fall in the cost distribution? | PERCENTILE_CONT, NTILE, RANK window functions |
| `care_gap_detection` | Which diabetic beneficiaries had no physician visit in a given year? | LEFT JOIN anti-join pattern |
| `utilization_trends` | How are claim volume and costs trending year over year? | LAG window function over year partitions |

With the minimal fixture data above, most queries return 1 row — the interesting patterns emerge with the full 20-subsample dataset (~2.3M beneficiaries).

In [ ]:
from cms_platform.analytics import queries

query_fns = [
    ("readmission_30day",    queries.readmission_30day),
    ("cohort_segmentation",  queries.cohort_segmentation),
    ("cost_benchmarking",    queries.cost_benchmarking),
    ("care_gap_detection",   queries.care_gap_detection),
    ("utilization_trends",   queries.utilization_trends),
]

for name, fn in query_fns:
    df = fn(conn)
    print(f"\n--- {name} ({len(df)} rows) ---")
    if len(df) > 0:
        print(df)

---

## Section 4: Risk Stratification

The risk model predicts next-year high-cost status (top quartile by total spend) using prior-year features from `dim_beneficiary`.

**Pipeline:**
1. Extract 14 features per beneficiary: 11 chronic-condition boolean flags + 3 Medicare reimbursement amounts (IP, OP, carrier).
2. StandardScaler → LogisticRegression (interpretable; swap-in point for LightGBM or XGBoost).
3. `predict_risk()` returns a Polars Series of float scores in [0.0, 1.0].

**Why logistic regression?** Interpretable coefficients map directly to clinical feature importance — a requirement when presenting to compliance and clinical teams. The pipeline object makes the estimator swap trivial.

**Features (`RISK_FEATURES`):** `sp_alzheimer`, `sp_chf`, `sp_chrnkidn`, `sp_cncr`, `sp_copd`, `sp_depressn`, `sp_diabetes`, `sp_ischmcht`, `sp_osteoprs`, `sp_ra_oa`, `sp_strketia`, `medreimb_ip`, `medreimb_op`, `medreimb_car`

> **Note: synthetic data caps real predictive signal. These figures demonstrate the pipeline, not clinical validity.**

In [ ]:
import random
import polars as pl
from cms_platform.scoring.risk_model import train_risk_model, predict_risk, RISK_FEATURES
from cms_platform.common.config import Settings

random.seed(42)
n = 60
bool_cols = [c for c in RISK_FEATURES if not c.startswith("medreimb_")]
money_cols = [c for c in RISK_FEATURES if c.startswith("medreimb_")]

# Generate synthetic training features
data = {col: [random.random() > 0.7 for _ in range(n)] for col in bool_cols}
data.update({col: [round(random.uniform(0, 8000), 2) for _ in range(n)] for col in money_cols})
features = pl.DataFrame(data)

# Label: high-risk if >=3 chronic conditions OR any reimbursement > $5,000
target = pl.Series("label", [
    int(sum(row[c] for c in bool_cols) >= 3 or any(row[c] > 5000 for c in money_cols))
    for row in features.to_dicts()
])

model = train_risk_model(features, target, Settings())
scores = predict_risk(model, features)

print(f"Training size:        {model.training_size} beneficiaries")
print(f"Scores (first 5):     {[round(s,3) for s in scores[:5].to_list()]}")
print(f"Score range:          [{scores.min():.3f}, {scores.max():.3f}]")
print(f"High-risk (>0.5):     {int((scores > 0.5).sum())}/{n}")
print()
print("Note: synthetic data caps real predictive signal.")
print("These figures demonstrate the pipeline, not clinical validity.")

---

## Section 5: Care-Gap Explanation (Ollama)

Care gaps (e.g., diabetic patient with no HbA1c in 90 days) are identified analytically via `care_gap_detection.sql`. The explainer module then generates a natural-language care coordinator summary via Ollama.

**Integration design:**
- Uses OpenAI-compatible SDK pointed at `settings.ollama_base_url` (default: `http://localhost:11434/v1`).
- Model is configurable via `settings.ollama_model` (default: `llama3.2`).
- **Stub fallback:** if Ollama is unreachable (connection refused, timeout, any exception), the function returns a deterministic rule-based summary. The demo runs fully offline — no LLM infrastructure required.
- The prompt is constrained: no invented medical details, 2–3 clinical sentences.

This design reflects a key principle for AI-driven features in regulated environments: **the LLM is an enhancement, never a hard dependency**.

In [ ]:
from cms_platform.scoring.explainer import explain_care_gaps
from cms_platform.common.config import Settings

gaps = [
    "No carrier claim in past 12 months (diabetic patient)",
    "Missing HbA1c within 90 days",
]
result = explain_care_gaps("BENE001", gaps, Settings())

print(f"Beneficiary:  {result.beneficiary_id}")
print(f"Model used:   {result.model_used}")
print(f"Open gaps:    {len(result.gaps)}")
print()
print("Summary:")
print(result.summary)

---

## Section 6: How This Scales — V0 → V2

Every V0 architectural choice was made with a concrete V2 swap point in mind. The repo carries one-liner seam annotations at the exact code locations where the upgrade happens.

### Swap-Point Table

| Swap point | V0 | V2 |
|------------|----|----||
| Storage | DuckDB embedded (single file) | Postgres + columnar warehouse (Redshift/BigQuery) |
| Ingestion | `ingest/load.py` chunked CSV reader | Kafka consumer (1 topic per claim type: `cms.inpatient`, `cms.carrier`, …) |
| API serving | Single Docker container | FastAPI behind load balancer + read replicas |
| Partitioning | None (all data in one DuckDB file) | `beneficiary_id_hash % N` shards — intra-shard joins never cross node boundaries |
| PHI audit | `audit.py` → Python structured logger | Dedicated Kafka topic → SIEM (Splunk/Datadog) |
| Schema registry | Implicit (code-defined column lists) | Confluent Schema Registry with Avro |

### V2 Seam Annotations in the Code

These are the **only** design comments in V0 code — each marks an exact upgrade location:

**`ingest/load.py`** (Kafka replacement seam):
```python
# V2 swap point: replace chunked CSV reader with a Kafka consumer here
conn = get_connection(settings)
```

**`schema/transforms.py`** (partitioning strategy):
```python
# V2 swap point: partitioning strategy note — at V2, each fact table will be
# hash-partitioned by (claim_year, beneficiary_id_hash % N) so intra-shard
# joins never cross node boundaries.
```

**`common/db.py`** (Postgres replacement seam):
```python
# V2 swap point: replace with Postgres via psycopg2 / asyncpg when migrating off DuckDB
```

### Why This Matters

The discipline is not architectural astronautics — it is about making tomorrow's migration **a refactor, not a rewrite**. Each seam annotation is a contract: swap the implementation behind this line, and the rest of the system does not change. This is the difference between a prototype and a platform.

---

## Closing: Repo Structure

```
cms-claims-demo/
├── src/cms_platform/
│   ├── ingest/          # WP1: download.py + load.py (explicit schemas, idempotent)
│   ├── schema/          # WP2: transforms.py (star schema, NOT EXISTS guards)
│   ├── analytics/       # WP3: queries.py + sql/analytics/ (5 query types)
│   ├── scoring/         # WP4: risk_model.py + explainer.py (sklearn + Ollama)
│   ├── api/             # WP5: FastAPI routes (4 endpoints, PHI masking at boundary)
│   └── common/          # audit.py, mask.py, config.py, db.py, logging.py
├── sql/
│   ├── schema/ddl.sql   # Star schema DDL
│   └── analytics/       # 5 analytical SQL files (each with header block)
├── tests/               # 76+ tests across all modules
├── .github/workflows/   # WP7: CI (quality → test → Docker)
├── ARCHITECTURE.md      # V2 distributed-systems design
├── COMPLIANCE.md        # PHI handling posture
└── notebooks/story.ipynb  # This notebook
```

See [`ARCHITECTURE.md`](../ARCHITECTURE.md) for the full V2 distributed-systems design: Postgres migration path, Kafka consumer design, HA topology, schema registry, and security posture.

---

> *Note: synthetic data caps real predictive signal. These figures demonstrate the pipeline, not clinical validity.*